# NER Model Playground

Тестирование NER детектора на произвольном тексте.

Поддерживаемые бэкенды:
- **`hf`** — HuggingFace transformers (по умолчанию: `Davlan/bert-base-multilingual-cased-ner-hrl`)
- **`gliner`** — GLiNER (по умолчанию: `urchade/gliner_medium-v2.1`)

## Конфигурация

In [44]:
# ── Настройки модели ──────────────────────────────────────────────
BACKEND = "hf"  # "hf" | "gliner"

HF_MODEL    = "Babelscape/wikineural-multilingual-ner"
GLINER_MODEL = "urchade/gliner_medium-v2.1"

DEVICE      = "cpu"   # "cpu" | "cuda" | "mps"
MIN_SCORE   = 0.3
ENTITY_TYPES = ["PER", "ORG"]  # PER=персоны, ORG=организации

## Загрузка модели

In [46]:
import sys, os
# Добавляем корень проекта в PYTHONPATH (если запускаем из папки notebooks/)
sys.path.insert(0, os.path.abspath(".."))

from repo_sanitizer.rulepack import NERConfig
from repo_sanitizer.detectors.ner import NERDetector, GLINER_LABEL_MAP

MODEL = GLINER_MODEL if BACKEND == "gliner" else HF_MODEL

config = NERConfig(
    model=MODEL,
    min_score=MIN_SCORE,
    entity_types=ENTITY_TYPES,
    device=DEVICE,
    backend=BACKEND,
)
detector = NERDetector(config)

# Прогреть модель
if BACKEND == "gliner":
    detector._ensure_gliner()
else:
    detector._ensure_pipeline()

print(f"Модель загружена: {MODEL} (backend={BACKEND}, device={DEVICE})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 958.56it/s, Materializing param=classifier.weight]                                       
BertForTokenClassification LOAD REPORT from: Babelscape/wikineural-multilingual-ner
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Модель загружена: Babelscape/wikineural-multilingual-ner (backend=hf, device=cpu)


## Инференс на произвольном тексте

In [42]:
TEXT = """
Elon Musk Илон Маск основал компанию SpaceX в 2002 году.
Сатья Наделла является генеральным директором Microsoft с 2014 года.
Джон Смит написал письмо в Европейский центральный банк.
Встреча состоялась в штаб-квартире Google в Маунтин-Вью.
Сергей Анчутин является основателем Doubletapp.

Владос дружит с Софой. 
+79022730219
savenvkov.vlad.vrx5
"""

In [43]:
from repo_sanitizer.detectors.base import Zone, ScanTarget

target = ScanTarget(
    file_path="<notebook>",
    content=TEXT,
    zones=[Zone(start=0, end=len(TEXT))],
)

findings = detector.detect(target)

if findings:
    print(f"Найдено сущностей: {len(findings)}\n")
    for f in findings:
        snippet = TEXT[max(0, f.offset_start - 10):f.offset_end + 10].replace("\n", "↵")
        print(f"  [{f.category.value:10s}] {f.matched_value!r:30s}  строка {f.line}  «…{snippet}…»")
else:
    print("Сущностей не найдено.")

Найдено сущностей: 12

  [ORG_NAME  ] 'Elon Musk'                     строка 2  «…↵Elon Musk Илон Маск…»
  [PII       ] 'Илон Маск'                     строка 2  «…Elon Musk Илон Маск основал к…»
  [ORG_NAME  ] 'SpaceX'                        строка 2  «… компанию SpaceX в 2002 го…»
  [PII       ] 'Сатья Наделла'                 строка 3  «…002 году.↵Сатья Наделла является …»
  [ORG_NAME  ] 'Microsoft'                     строка 3  «…иректором Microsoft с 2014 го…»
  [PII       ] 'Джон Смит'                     строка 4  «…014 года.↵Джон Смит написал п…»
  [ORG_NAME  ] 'Европейский центральный банк'  строка 4  «… письмо в Европейский центральный банк.↵Встреча …»
  [ORG_NAME  ] 'Google'                        строка 5  «…-квартире Google в Маунтин…»
  [PII       ] 'Сергей Анчутин'                строка 6  «…нтин-Вью.↵Сергей Анчутин является …»
  [ORG_NAME  ] 'Doubletapp'                    строка 6  «…нователем Doubletapp.↵↵Владос …»
  [PII       ] 'Владос'                        строка

## Прямой вызов модели (без фильтров детектора)

In [47]:
# Смотрим «сырой» вывод модели — все сущности без порога и типов
raw = detector._infer_batch([TEXT])[0]

if raw:
    print(f"{'Тип':12s} {'Слово':35s} {'Score':6s}  Позиция")
    print("-" * 65)
    for e in raw:
        print(f"{e['entity_group']:12s} {e['word']!r:35s} {e['score']:.3f}  [{e['start']}:{e['end']}]")
else:
    print("Сырых сущностей нет.")

Тип          Слово                               Score   Позиция
-----------------------------------------------------------------
ORG          'Elon Musk'                         0.721  [1:10]
PER          'Илон Маск'                         0.906  [11:20]
ORG          'SpaceX'                            0.987  [38:44]
PER          'Сатья Наделла'                     0.995  [58:71]
ORG          'Microsoft'                         0.986  [104:113]
PER          'Джон Смит'                         0.995  [127:136]
ORG          'Европейский центральный банк'      0.922  [154:182]
ORG          'Google'                            0.966  [219:225]
LOC          'М'                                 0.998  [228:229]
LOC          '##аунтин - Вью'                    0.983  [229:239]
PER          'Сергей Анчутин'                    0.995  [241:255]
ORG          'Doubletapp'                        0.889  [277:287]
PER          'В'                                 0.983  [290:291]
PER          '##ладо

## Интерактивный ввод текста

In [32]:
def run_ner(text: str, min_score: float = MIN_SCORE) -> None:
    """Запустить NER на произвольном тексте и красиво вывести результат."""
    chunks = detector._chunk_text(text)
    chunk_texts = [c for _, c in chunks]
    raw_results = detector._infer_batch(chunk_texts)

    print(f"Текст ({len(text)} симв.) разбит на {len(chunks)} чанк(ов)\n")

    found_any = False
    for (offset, chunk), raw in zip(chunks, raw_results):
        for e in raw:
            if e["score"] < min_score:
                continue
            found_any = True
            abs_start = offset + e["start"]
            abs_end   = offset + e["end"]
            ctx = text[max(0, abs_start - 15):abs_end + 15].replace("\n", "↵")
            print(
                f"  [{e['entity_group']:4s}] {e['score']:.2f}  "
                f"{e['word']!r:30s}  pos [{abs_start}:{abs_end}]  «…{ctx}…»"
            )

    if not found_any:
        print("Ничего не найдено (попробуйте снизить min_score).")


# ── Введите свой текст ────────────────────────────────────────────
run_ner("""
Dear Mr. Johnson, please forward this to the team at Acme Corp.
Regards, Alice Brown.
""")

Текст (87 симв.) разбит на 1 чанк(ов)

  [PER ] 0.92  'Johnson'                       pos [10:17]  «…↵Dear Mr. Johnson, please forwar…»
  [ORG ] 0.95  'Acme Corp'                     pos [54:63]  «…to the team at Acme Corp.↵Regards, Alic…»
  [ORG ] 0.47  'Reg'                           pos [65:68]  «… at Acme Corp.↵Regards, Alice Bro…»
  [PER ] 0.93  'Alice Brown'                   pos [74:85]  «…Corp.↵Regards, Alice Brown.↵…»


## Батчевое тестирование

In [ ]:
SAMPLES = [
    "Barack Obama served as the 44th President of the United States.",
    "Sergei Ivanov works at Yandex in Moscow.",
    "The report was sent to the European Commission by Dr. Maria Schmidt.",
    "No personal information in this line — just some code: x = 42",
    "Elon Musk and Tim Cook attended the conference organized by Apple.",
]

raw_batch = detector._infer_batch(SAMPLES)

for text, entities in zip(SAMPLES, raw_batch):
    filtered = [e for e in entities if e["score"] >= MIN_SCORE]
    tag = ", ".join(f"{e['entity_group']}:{e['word']!r}" for e in filtered) or "—"
    print(f"{text[:60]:<60s}  →  {tag}")